EDA

In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

STORAGE_PATH = Path("../datasets/processed/lower48_weekly_storage_latest.parquet")

storage = pd.read_parquet(STORAGE_PATH)

In [ ]:
# Prepared in notebooks/01a_data_preparation.ipynb
storage = storage.sort_values("date", ascending=True).reset_index(drop=True)

storage.tail()


,date,storage_bcf,weekly_change_bcf,year,month,week_of_year,duoarea
854,2026-05-22,2483,92.0,2026,5,21,R48
855,2026-05-29,2578,95.0,2026,5,22,R48
856,2026-06-05,2686,108.0,2026,6,23,R48
857,2026-06-12,2759,73.0,2026,6,24,R48
858,2026-06-19,2835,76.0,2026,6,25,R48


In [6]:
#plot total storage over time

import plotly.express as px

plot_df = storage.sort_values('date')

fig = px.line(
    plot_df,
    x='date',
    y='storage_bcf',
    title='Total Storage in BCF',
    labels={'date': 'Date', 'storage_bcf': 'Storage (BCF)'},
)

minima = plot_df.loc[plot_df['storage_bcf'].idxmin()]
maxima = plot_df.loc[plot_df['storage_bcf'].idxmax()]
min_val = minima['storage_bcf']
max_val = maxima['storage_bcf']

fig.add_hline(
    y=min_val,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Min: {min_val:.0f} BCF',
    annotation_position='bottom left',
)
fig.add_hline(
    y=max_val,
    line_dash='dash',
    line_color='green',
    annotation_text=f'Max: {max_val:.0f} BCF',
    annotation_position='top left',
)

fig.add_scatter(
    x=[minima['date']],
    y=[min_val],
    mode='markers',
    name=f"Min ({minima['date'].strftime('%b %Y')})",
    marker=dict(size=12, color='red', symbol='circle'),
)
fig.add_scatter(
    x=[maxima['date']],
    y=[max_val],
    mode='markers',
    name=f"Max ({maxima['date'].strftime('%b %Y')})",
    marker=dict(size=12, color='green', symbol='circle'),
)

fig.update_xaxes(
    tickformat='%b %Y',
    dtick='M3',
    tickangle=-45,
)

fig.show()

#exhibits seasonal patterns in storage, with highs in winter season which is then used into the spring season, with buildup during summer and autumn seasons

In [7]:
#plot weekly diff over time

plot_df = storage.sort_values('date')

fig = px.line(
    plot_df,
    x='date',
    y='weekly_change_bcf',
    title='Weekly Change in Storage (in BCF)',
    labels={'date': 'Date', 'weekly_change_bcf': 'Weekly Change (BCF)'},
)

minima = plot_df.loc[plot_df['weekly_change_bcf'].idxmin()]
maxima = plot_df.loc[plot_df['weekly_change_bcf'].idxmax()]
min_val = minima['weekly_change_bcf']
max_val = maxima['weekly_change_bcf']

fig.add_hline(
    y=min_val,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Min: {min_val:.0f} BCF',
    annotation_position='bottom left',
)
fig.add_hline(
    y=max_val,
    line_dash='dash',
    line_color='green',
    annotation_text=f'Max: {max_val:.0f} BCF',
    annotation_position='top left',
)

fig.add_scatter(
    x=[minima['date']],
    y=[min_val],
    mode='markers',
    name=f"Min ({minima['date'].strftime('%b %Y')})",
    marker=dict(size=12, color='red', symbol='circle'),
)
fig.add_scatter(
    x=[maxima['date']],
    y=[max_val],
    mode='markers',
    name=f"Max ({maxima['date'].strftime('%b %Y')})",
    marker=dict(size=12, color='green', symbol='circle'),
)

fig.update_xaxes(
    tickformat='%b %Y',
    dtick='M3',
    tickangle=-45,
)

fig.show()

# shows seasonality, with large injections during the summer seasons and drawdowns during winter months. generally 8 months of injection, then 4 months of withdrawal. withdrawals are generally larger per week for the 4 month periods


In [8]:
#graphing weekly avg (averaged out over week of year)
weekly_stats = (
    storage
    .groupby("week_of_year")
    .agg(
        avg=("weekly_change_bcf", "mean"),
        std=("weekly_change_bcf", "std")
    )
    .reset_index()
)

weekly_stats.head()

fig2=px.line(weekly_stats, x='week_of_year', y='avg',title='Average Weekly Change in Storage (BCF) by Week of Year')

fig2.add_scatter(
    x=weekly_stats['week_of_year'],
    y=weekly_stats['avg'] + weekly_stats['std'],
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip',
)
fig2.add_scatter(
    x=weekly_stats['week_of_year'],
    y=weekly_stats['avg'] - weekly_stats['std'],
    mode='lines',
    fill='tonexty',
    fillcolor='rgba(0, 100, 255, 0.2)',
    name='± 1 std',
    hoverinfo='skip',
)


fig2.show( )



In [9]:
# all years overlaid — weekly change by week of year

import plotly.express as px

overlay_df = (
    storage
    .dropna(subset=['weekly_change_bcf'])
    .sort_values(['year', 'week_of_year'])
    .copy()
)
overlay_df['year'] = overlay_df['year'].astype(str)

fig3 = px.line(
    overlay_df,
    x='week_of_year',
    y='weekly_change_bcf',
    color='year',
    title='Weekly Change in Storage by Week of Year (All Years Overlaid)',
    labels={
        'week': 'Week of Year',
        'weekly_change_bcf': 'Weekly Change (BCF)',
        'year': 'Year',
    },
)

fig3.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1)
fig3.update_xaxes(dtick=4, range=[1, 53])
fig3.update_layout(legend_title_text='Year')
fig3.show()

In [10]:
# weekly injection distribution

fig4=px.histogram(storage, x='weekly_change_bcf', title='Weekly Change in Storage (in BCF)')

fig4.show()

storage["weekly_change_bcf"].describe()


#heavily left skewed, with most weeks showing injections of 60-80 BCF with a few weeks of larger drawdowns

count    859.000000
mean      -0.328289
std       99.079054
min     -359.000000
25%      -62.000000
50%       35.000000
75%       76.000000
max      132.000000
Name: weekly_change_bcf, dtype: float64

In [11]:
# boxplot by month

fig5=px.box(storage, x='month', y='weekly_change_bcf', title='Weekly Change in Storage (in BCF) by Month')
fig5.show()

In [12]:
#rolling average (1 month)

rolling_4_week = storage['weekly_change_bcf'].rolling(window=4).mean()

fig6=px.line(storage, x='date', y=rolling_4_week, title='Rolling Average of Weekly Change in Storage (in BCF) (4 Week Window)')
fig6.show()

#plot weekly change by year and 4 week rolling average


In [13]:
# rolling average (4 week) overlaid on raw weekly change (smoothing for visual clarity)
plot_df = storage.sort_values('date').copy()
plot_df['rolling_4_week'] = plot_df['weekly_change_bcf'].rolling(window=4).mean()
fig6 = px.line(
    plot_df,
    x='date',
    y='weekly_change_bcf',
    title='Weekly Change vs 4-Week Rolling Average (BCF)',
    labels={'date': 'Date', 'weekly_change_bcf': 'Weekly Change (BCF)'},
)
fig6.update_traces(
    name='Weekly Change',
    line=dict(color='darkgray', width=1),
    opacity=0.55,
)
fig6.add_scatter(
    x=plot_df['date'],
    y=plot_df['rolling_4_week'],
    mode='lines',
    name='4-Week Rolling Avg',
    line=dict(color='#636efa', width=2.5),
)
fig6.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1)
fig6.update_xaxes(tickformat='%b %Y', dtick='M6', tickangle=-45)
fig6.show()


In [14]:
#summary stats

max_storage = storage['storage_bcf'].max()
min_storage = storage['storage_bcf'].min()
max_change = storage['weekly_change_bcf'].max()
min_change = storage['weekly_change_bcf'].min()

print(f"Maximum Storage: {max_storage:.0f} BCF")
print(f"Minimum Storage: {min_storage:.0f} BCF")
print(f"Maximum Change: {max_change:.0f} BCF")
print(f"Minimum Change: {min_change:.0f} BCF")

mean = storage['weekly_change_bcf'].mean()
print(f"Mean Weekly Change: {mean:.0f} BCF")
median = storage['weekly_change_bcf'].median()
print(f"Median Weekly Change: {median:.0f} BCF")


Maximum Storage: 4047 BCF
Minimum Storage: 824 BCF
Maximum Change: 132 BCF
Minimum Change: -359 BCF
Mean Weekly Change: -0 BCF
Median Weekly Change: 35 BCF


In [15]:
# 5-year seasonal baseline (prior 5 complete years, excluding current)

latest_year = int(storage["year"].max())
baseline_start = latest_year - 5
baseline_end = latest_year - 1
baseline_label = f"{baseline_start}–{baseline_end}"

baseline = storage[storage["year"].between(baseline_start, baseline_end)].copy()

# Equal weight per year: average each year's profile, then average across years
yearly_profiles = (
    baseline
    .dropna(subset=["weekly_change_bcf"])
    .groupby(["year", "week_of_year"], as_index=False)["weekly_change_bcf"]
    .mean()
)

five_year_avg = (
    yearly_profiles
    .groupby("week_of_year")
    .agg(
        avg_change=("weekly_change_bcf", "mean"),
        std_change=("weekly_change_bcf", "std"),
        min_change=("weekly_change_bcf", "min"),
        max_change=("weekly_change_bcf", "max"),
    )
    .reset_index()
    .sort_values("week_of_year")
)

fig7 = go.Figure()
fig7.add_trace(go.Scatter(
    x=five_year_avg["week_of_year"],
    y=five_year_avg["avg_change"] + five_year_avg["std_change"],
    mode="lines",
    line=dict(width=0),
    showlegend=False,
    hoverinfo="skip",
))
fig7.add_trace(go.Scatter(
    x=five_year_avg["week_of_year"],
    y=five_year_avg["avg_change"] - five_year_avg["std_change"],
    mode="lines",
    line=dict(width=0),
    fill="tonexty",
    fillcolor="rgba(99, 110, 250, 0.2)",
    name="±1σ range",
    hoverinfo="skip",
))
fig7.add_trace(go.Scatter(
    x=five_year_avg["week_of_year"],
    y=five_year_avg["avg_change"],
    mode="lines",
    name="5-yr avg",
    line=dict(color="#636efa", width=2),
))
fig7.update_layout(
    title=f"5-Year Average Weekly Storage Change ({baseline_label}, BCF)",
    xaxis_title="Week of Year",
    yaxis_title="Avg Weekly Change (BCF)",
)
fig7.show()

In [16]:
# Current year vs 5-year seasonal baseline

current_year = storage[storage["year"] == latest_year].copy()

current_vs_avg = (
    current_year
    .merge(five_year_avg, on="week_of_year", how="left")
    .sort_values("week_of_year")
)

current_vs_avg["change_vs_5yr_avg"] = (
    current_vs_avg["weekly_change_bcf"] - current_vs_avg["avg_change"]
)
current_vs_avg["cumulative_vs_5yr_avg"] = current_vs_avg["change_vs_5yr_avg"].cumsum()

# Weekly deviation from norm
fig8 = px.line(
    current_vs_avg,
    x="week_of_year",
    y="change_vs_5yr_avg",
    title=f"{latest_year} Weekly Change vs 5-Year Average ({baseline_label}, BCF)",
    labels={
        "week_of_year": "Week of Year",
        "change_vs_5yr_avg": "Deviation from 5-Yr Avg (BCF)",
    },
)
fig8.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
fig8.show()

# Current year vs average with ±1σ band
fig9 = go.Figure()
fig9.add_trace(go.Scatter(
    x=five_year_avg["week_of_year"],
    y=five_year_avg["avg_change"] + five_year_avg["std_change"],
    mode="lines",
    line=dict(width=0),
    showlegend=False,
    hoverinfo="skip",
))
fig9.add_trace(go.Scatter(
    x=five_year_avg["week_of_year"],
    y=five_year_avg["avg_change"] - five_year_avg["std_change"],
    mode="lines",
    line=dict(width=0),
    fill="tonexty",
    fillcolor="rgba(99, 110, 250, 0.2)",
    name="±1σ range",
    hoverinfo="skip",
))
fig9.add_trace(go.Scatter(
    x=five_year_avg["week_of_year"],
    y=five_year_avg["avg_change"],
    mode="lines",
    name=f"5-yr avg ({baseline_label})",
    line=dict(color="#636efa", width=2),
))
fig9.add_trace(go.Scatter(
    x=current_vs_avg["week_of_year"],
    y=current_vs_avg["weekly_change_bcf"],
    mode="lines+markers",
    name=str(latest_year),
    line=dict(color="#EF553B", width=2),
))
fig9.update_layout(
    title=f"{latest_year} vs 5-Year Average Weekly Storage Change (BCF)",
    xaxis_title="Week of Year",
    yaxis_title="Weekly Change (BCF)",
)
fig9.show()

# Cumulative surplus/deficit vs normal season
fig10 = px.line(
    current_vs_avg,
    x="week_of_year",
    y="cumulative_vs_5yr_avg",
    title=f"{latest_year} Cumulative Deviation from 5-Year Norm ({baseline_label}, BCF)",
    labels={
        "week_of_year": "Week of Year",
        "cumulative_vs_5yr_avg": "Cumulative vs 5-Yr Avg (BCF)",
    },
)
fig10.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
fig10.show()

# this one shows the differential in injection/withdrawal vs the 5 year average. note that current year is not complete, so not included in the rolling 5 year average

In [17]:
# Build yearly vs 5-year average storage profiles

yearly_storage_profiles = (
    baseline
    .groupby(["year", "week_of_year"], as_index=False)["storage_bcf"]
    .mean()
)

five_year_avg_storage = (
    yearly_storage_profiles
    .groupby("week_of_year")
    .agg(
        avg_storage=("storage_bcf", "mean"),
        std_storage=("storage_bcf", "std"),
        min_storage=("storage_bcf", "min"),
        max_storage=("storage_bcf", "max"),
    )
    .reset_index()
    .sort_values("week_of_year")
)

current_vs_avg_all = (
    current_year
    .merge(five_year_avg, on="week_of_year", how="left")
    .merge(five_year_avg_storage, on="week_of_year", how="left")
    .sort_values("week_of_year")
)

current_vs_avg_all["storage_vs_5yr_avg"] = (
    current_vs_avg_all["storage_bcf"] - current_vs_avg_all["avg_storage"]
)

current_vs_avg_all["weekly_change_deviation_bcf"] = current_vs_avg_all["weekly_change_bcf"] - current_vs_avg_all["avg_change"]

current_vs_avg_all.head()

,date,storage_bcf,weekly_change_bcf,year,month,week_of_year,duoarea,avg_change,std_change,min_change,max_change,avg_storage,std_storage,min_storage,max_storage,storage_vs_5yr_avg,weekly_change_deviation_bcf
0,2026-01-02,3244,-120.0,2026,1,1,R48,-96.4,78.824489,-179.0,11.0,3164.6,202.999507,2902.0,3373.0,79.4,-23.6
1,2026-01-09,3174,-70.0,2026,1,2,R48,-177.4,65.297779,-258.0,-82.0,2987.2,168.901451,2810.0,3182.0,186.8,107.4
2,2026-01-16,3054,-120.0,2026,1,3,R48,-196.4,93.382547,-326.0,-86.0,2790.8,128.232991,2591.0,2892.0,263.2,76.4
3,2026-01-23,2813,-241.0,2026,1,4,R48,-225.8,67.843202,-321.0,-151.0,2565.0,144.166570,2323.0,2689.0,248.0,-15.2
4,2026-01-30,2454,-359.0,2026,1,5,R48,-171.8,59.031348,-222.0,-75.0,2393.2,185.818998,2101.0,2584.0,60.8,-187.2


In [19]:
# Current year total inventory vs 5-year average storage profile

fig11 = go.Figure()
fig11.add_trace(go.Scatter(
    x=current_vs_avg_all["week_of_year"],
    y=current_vs_avg_all["avg_storage"] + current_vs_avg_all["std_storage"],
    mode="lines",
    line=dict(width=0),
    showlegend=False,
    hoverinfo="skip",
))
fig11.add_trace(go.Scatter(
    x=current_vs_avg_all["week_of_year"],
    y=current_vs_avg_all["avg_storage"] - current_vs_avg_all["std_storage"],
    mode="lines",
    line=dict(width=0),
    fill="tonexty",
    fillcolor="rgba(99, 110, 250, 0.2)",
    name="±1σ range",
    hoverinfo="skip",
))
fig11.add_trace(go.Scatter(
    x=current_vs_avg_all["week_of_year"],
    y=current_vs_avg_all["avg_storage"],
    mode="lines",
    name=f"5-yr avg storage ({baseline_label})",
    line=dict(color="#636efa", width=2),
))
fig11.add_trace(go.Scatter(
    x=current_vs_avg_all["week_of_year"],
    y=current_vs_avg_all["storage_bcf"],
    mode="lines+markers",
    name=str(latest_year),
    line=dict(color="#EF553B", width=2),
))
fig11.update_layout(
    title=f"{latest_year} Total Storage vs 5-Year Average ({baseline_label}, BCF)",
    xaxis_title="Week of Year",
    yaxis_title="Storage (BCF)",
)
fig11.show()

Total storage started above 5 year averages, but later dropped below 5 year averages due to inclement weather in winter months. Storage has generally now stabilized above historicals.

In [23]:
# identify weeks in current year that are outside of 5 year range

current_vs_avg_all["outside_5yr_range"] = (
    (current_vs_avg_all["weekly_change_bcf"]
     < current_vs_avg_all["min_change"])
    |
    (current_vs_avg_all["weekly_change_bcf"]
     > current_vs_avg_all["max_change"])
)

print(current_vs_avg_all["outside_5yr_range"].value_counts())

print(current_vs_avg_all.loc[current_vs_avg_all["outside_5yr_range"], ["date", "week_of_year", "weekly_change_bcf"]])

outside_5yr_range
False    14
True     11
Name: count, dtype: int64
         date  week_of_year  weekly_change_bcf
1  2026-01-09             2              -70.0
4  2026-01-30             5             -359.0
5  2026-02-06             6             -252.0
7  2026-02-20             8              -52.0
8  2026-02-27             9             -131.0
10 2026-03-13            11               34.0
11 2026-03-20            12              -54.0
12 2026-03-27            13               33.0
15 2026-04-17            16              103.0
17 2026-05-01            18               63.0
21 2026-05-29            22               95.0
